---
## ✏️ Exercise 2: Train Word2Vec on a Larger Corpus

We build a 20-sentence product/tech review corpus, preprocess it fully,
then train and compare two Word2Vec architectures:

| Architecture | How it works | Best for |
|---|---|---|
| **CBOW** | Predicts target word from surrounding context | Frequent words, faster training |
| **Skip-gram** | Predicts context words from a target word | Rare words, richer embeddings |

In [1]:
import re, string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

raw_corpus = [
    "The smartphone has an incredible display with vivid colors and sharp resolution.",
    "Battery life is outstanding and lasts more than two days on a single charge.",
    "The camera produces stunning photographs even in low light conditions.",
    "Customer service was very helpful and resolved my issue within minutes.",
    "The laptop keyboard feels comfortable and typing experience is smooth.",
    "Delivery was fast and the packaging was secure preventing any damage.",
    "The sound quality of these headphones is rich, deep, and immersive.",
    "Screen resolution on this monitor is crystal clear and eye friendly.",
    "The gaming performance of this GPU is exceptional at high settings.",
    "This smartwatch tracks fitness data accurately including heart rate and sleep.",
    "Software updates are frequent and the operating system runs smoothly.",
    "The build quality feels premium with a solid aluminum chassis design.",
    "Connectivity options include USB-C, HDMI, and fast wireless Bluetooth.",
    "The streaming service offers thousands of movies and series in HD quality.",
    "Setup was straightforward and the user manual explains every step clearly.",
    "Price is reasonable compared to competitors offering similar specifications.",
    "The touchscreen response is highly accurate with minimal input latency.",
    "Voice assistant integration works seamlessly with smart home devices.",
    "Return policy is flexible and refunds are processed within three days.",
    "Overall this product exceeded my expectations and I highly recommend it.",
]

print(f"Corpus size: {len(raw_corpus)} sentences")

Corpus size: 20 sentences


In [2]:
#lowercase -> digits -> punctuation -> tokenize -> stopwords -> lemmatize
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(w) for w in tokens
              if w not in stop_words and len(w) > 1]
    return tokens

tokenized_corpus = [preprocess(sent) for sent in raw_corpus]

print("Tokenized corpus (first 5 sentences):")
for i, t in enumerate(tokenized_corpus[:5]):
    print(f"  {i+1}: {t}")

total_tokens = sum(len(s) for s in tokenized_corpus)
print(f"\nTotal sentences : {len(tokenized_corpus)}")
print(f"Total tokens    : {total_tokens}")

Tokenized corpus (first 5 sentences):
  1: ['smartphone', 'incredible', 'display', 'vivid', 'color', 'sharp', 'resolution']
  2: ['battery', 'life', 'outstanding', 'last', 'two', 'day', 'single', 'charge']
  3: ['camera', 'produce', 'stunning', 'photograph', 'even', 'low', 'light', 'condition']
  4: ['customer', 'service', 'helpful', 'resolved', 'issue', 'within', 'minute']
  5: ['laptop', 'keyboard', 'feel', 'comfortable', 'typing', 'experience', 'smooth']

Total sentences : 20
Total tokens    : 146


In [3]:
# Train CBOW model (sg=0)
cbow_model = Word2Vec(
    tokenized_corpus,
    vector_size=100,
    window=5,
    min_count=1,
    sg=0,
    epochs=100,
    seed=42,
    workers=4
)

# Train Skip-gram model (sg=1)
sg_model = Word2Vec(
    tokenized_corpus,
    vector_size=100,
    window=5,
    min_count=1,
    sg=1,
    epochs=100,
    seed=42,
    workers=4
)

vocab = sorted(cbow_model.wv.index_to_key)
print(f"Vocabulary size : {len(vocab)}")
print(f"Sample words    : {vocab[:20]}")

Vocabulary size : 137
Sample words    : ['accurate', 'accurately', 'aluminum', 'assistant', 'battery', 'bluetooth', 'build', 'camera', 'charge', 'chassis', 'clear', 'clearly', 'color', 'comfortable', 'compared', 'competitor', 'condition', 'connectivity', 'crystal', 'customer']


In [4]:
# Most similar words — CBOW vs Skip-gram

probe_words = ['camera', 'battery', 'quality', 'screen']

print("Most Similar Words - CBOW (sg=0)")
print("-" * 55)
for word in probe_words:
    similar = cbow_model.wv.most_similar(word, topn=3)
    pairs = ', '.join([f"{w}({s:.2f})" for w, s in similar])
    print(f"  {word:<12} -> {pairs}")

print("\nMost Similar Words - Skip-gram (sg=1)")
print("-" * 55)
for word in probe_words:
    similar = sg_model.wv.most_similar(word, topn=3)
    pairs = ', '.join([f"{w}({s:.2f})" for w, s in similar])
    print(f"  {word:<12} -> {pairs}")

Most Similar Words - CBOW (sg=0)
-------------------------------------------------------
  camera       -> refund(0.32), update(0.30), input(0.30)
  battery      -> every(0.31), step(0.30), straightforward(0.29)
  quality      -> connectivity(0.34), processed(0.33), compared(0.32)
  screen       -> refund(0.37), crystal(0.29), chassis(0.29)

Most Similar Words - Skip-gram (sg=1)
-------------------------------------------------------
  camera       -> photograph(0.71), low(0.70), refund(0.68)
  battery      -> every(0.68), step(0.66), straightforward(0.65)
  quality      -> every(0.79), explains(0.78), premium(0.78)
  screen       -> refund(0.73), photograph(0.68), assistant(0.68)


In [5]:
# Cosine similarity
# Values range from -1 (opposite) to +1 (identical direction)

pairs = [
    ('camera',  'photograph'),
    ('battery', 'charge'),
    ('screen',  'display'),
    ('quality', 'premium'),
]

print(f"{'Word 1':<15} {'Word 2':<15} {'CBOW':>8} {'Skip-gram':>10}")
print("-" * 52)
for w1, w2 in pairs:
    cbow_sim = cbow_model.wv.similarity(w1, w2)
    sg_sim   = sg_model.wv.similarity(w1, w2)
    print(f"  {w1:<13} {w2:<15} {cbow_sim:>8.4f} {sg_sim:>10.4f}")

Word 1          Word 2              CBOW  Skip-gram
----------------------------------------------------
  camera        photograph        0.2288     0.7098
  battery       charge            0.0092     0.4645
  screen        display          -0.0753     0.4862
  quality       premium           0.2771     0.7761


---
## ✏️ Exercise 3: Cosine Similarity — Positive vs Negative Words

We compare three types of pairings to test whether Word2Vec clusters
sentiment-bearing words correctly:

| Comparison | Expected behaviour |
|---|---|
| Positive ↔ Positive | **High** similarity (same sentiment cluster) |
| Negative ↔ Negative | **High** similarity (same sentiment cluster) |
| Positive ↔ Negative | **Lower** similarity (opposite sentiment poles) |

In [6]:
extra_sentences = [
    "The product is absolutely amazing and I love the excellent performance.",
    "Terrible experience, the device is horrible and completely broken.",
    "This is the best purchase I have ever made, truly outstanding quality.",
    "Worst product ever, totally disappointed and very frustrated with it.",
    "Fantastic build, the premium design looks beautiful and feels perfect.",
    "Awful customer support, the response was rude and completely unhelpful.",
    "The excellent display is bright, vivid, and delivers superb clarity.",
    "Very poor quality, the cheap materials feel fragile and look ugly.",
    "I am very happy and satisfied with this wonderful and brilliant device.",
    "Dreadful performance, the sluggish system crashes and freezes constantly.",
]

extended_corpus = raw_corpus + extra_sentences
tokenized_extended = [preprocess(sent) for sent in extended_corpus]

print(f"Original corpus  : {len(raw_corpus)} sentences")
print(f"Extended corpus  : {len(extended_corpus)} sentences")
print(f"Total tokens     : {sum(len(s) for s in tokenized_extended)}")

Original corpus  : 20 sentences
Extended corpus  : 30 sentences
Total tokens     : 213


In [7]:
sg_extended = Word2Vec(
    tokenized_extended,
    vector_size=100,
    window=5,
    min_count=1,
    sg=1,
    epochs=200,
    seed=42,
    workers=1
)

vocab = set(sg_extended.wv.index_to_key)
print(f"Vocabulary size  : {len(vocab)}")

positive_words = [w for w in
    ['amazing', 'excellent', 'outstanding', 'fantastic',
     'brilliant', 'superb', 'wonderful', 'perfect', 'beautiful', 'satisfied']
    if w in vocab]

negative_words = [w for w in
    ['terrible', 'horrible', 'awful', 'dreadful',
     'disappointed', 'frustrated', 'poor', 'worst', 'ugly', 'sluggish']
    if w in vocab]

print(f"Positive words   : {positive_words}")
print(f"Negative words   : {negative_words}")

Vocabulary size  : 180
Positive words   : ['amazing', 'excellent', 'outstanding', 'fantastic', 'brilliant', 'superb', 'wonderful', 'perfect', 'beautiful', 'satisfied']
Negative words   : ['terrible', 'horrible', 'awful', 'dreadful', 'disappointed', 'frustrated', 'poor', 'worst', 'ugly', 'sluggish']


In [8]:
# ── Positive vs Positive ─────────────────────────────────────────

import pandas as pd

pp_rows = []
for i in range(len(positive_words)):
    for j in range(i + 1, len(positive_words)):
        w1, w2 = positive_words[i], positive_words[j]
        pp_rows.append((w1, w2, round(sg_extended.wv.similarity(w1, w2), 4)))

pp_df = pd.DataFrame(pp_rows, columns=['Word 1', 'Word 2', 'Similarity'])
avg_pp = pp_df['Similarity'].mean()

print(f"Positive vs Positive  ({len(pp_rows)} pairs)")
print(pp_df.to_string(index=False))
print(f"\n  Average similarity: {avg_pp:.4f}")

Positive vs Positive  (45 pairs)
     Word 1      Word 2  Similarity
    amazing   excellent      0.9908
    amazing outstanding      0.9661
    amazing   fantastic      0.9653
    amazing   brilliant      0.9830
    amazing      superb      0.9839
    amazing   wonderful      0.9814
    amazing     perfect      0.9602
    amazing   beautiful      0.9513
    amazing   satisfied      0.9832
  excellent outstanding      0.9606
  excellent   fantastic      0.9585
  excellent   brilliant      0.9746
  excellent      superb      0.9930
  excellent   wonderful      0.9706
  excellent     perfect      0.9538
  excellent   beautiful      0.9452
  excellent   satisfied      0.9721
outstanding   fantastic      0.9577
outstanding   brilliant      0.9674
outstanding      superb      0.9585
outstanding   wonderful      0.9629
outstanding     perfect      0.9539
outstanding   beautiful      0.9453
outstanding   satisfied      0.9665
  fantastic   brilliant      0.9722
  fantastic      superb      0.

In [9]:
# ── Negative vs Negative ─────────────────────────────────────────

nn_rows = []
for i in range(len(negative_words)):
    for j in range(i + 1, len(negative_words)):
        w1, w2 = negative_words[i], negative_words[j]
        nn_rows.append((w1, w2, round(sg_extended.wv.similarity(w1, w2), 4)))

nn_df = pd.DataFrame(nn_rows, columns=['Word 1', 'Word 2', 'Similarity'])
avg_nn = nn_df['Similarity'].mean()

print(f"Negative vs Negative  ({len(nn_rows)} pairs)")
print(nn_df.to_string(index=False))
print(f"\n  Average similarity: {avg_nn:.4f}")

Negative vs Negative  (45 pairs)
      Word 1       Word 2  Similarity
    terrible     horrible      0.9957
    terrible        awful      0.9800
    terrible     dreadful      0.9802
    terrible disappointed      0.9759
    terrible   frustrated      0.9771
    terrible         poor      0.9696
    terrible        worst      0.9771
    terrible         ugly      0.9755
    terrible     sluggish      0.9673
    horrible        awful      0.9770
    horrible     dreadful      0.9768
    horrible disappointed      0.9738
    horrible   frustrated      0.9745
    horrible         poor      0.9629
    horrible        worst      0.9751
    horrible         ugly      0.9691
    horrible     sluggish      0.9646
       awful     dreadful      0.9672
       awful disappointed      0.9653
       awful   frustrated      0.9678
       awful         poor      0.9521
       awful        worst      0.9677
       awful         ugly      0.9605
       awful     sluggish      0.9531
    dreadful disa

In [10]:
# ── Positive vs Negative ─────────────────────────────────────────

pn_rows = []
for w1 in positive_words:
    for w2 in negative_words:
        pn_rows.append((w1, w2, round(sg_extended.wv.similarity(w1, w2), 4)))

pn_df = pd.DataFrame(pn_rows, columns=['Positive', 'Negative', 'Similarity'])
avg_pn = pn_df['Similarity'].mean()

print(f"Positive vs Negative  ({len(pn_rows)} pairs)")
print(pn_df.to_string(index=False))
print(f"\n  Average similarity: {avg_pn:.4f}")

Positive vs Negative  (100 pairs)
   Positive     Negative  Similarity
    amazing     terrible      0.9802
    amazing     horrible      0.9783
    amazing        awful      0.9734
    amazing     dreadful      0.9842
    amazing disappointed      0.9858
    amazing   frustrated      0.9873
    amazing         poor      0.9575
    amazing        worst      0.9875
    amazing         ugly      0.9649
    amazing     sluggish      0.9720
  excellent     terrible      0.9707
  excellent     horrible      0.9697
  excellent        awful      0.9686
  excellent     dreadful      0.9751
  excellent disappointed      0.9783
  excellent   frustrated      0.9804
  excellent         poor      0.9542
  excellent        worst      0.9804
  excellent         ugly      0.9606
  excellent     sluggish      0.9600
outstanding     terrible      0.9636
outstanding     horrible      0.9586
outstanding        awful      0.9490
outstanding     dreadful      0.9608
outstanding disappointed      0.9707
outs

In [13]:
# ── Summary: bar chart of average similarities ────────────────────

print(f"\n{'Category':<25} {'Avg Similarity':>15}")
print("-" * 42)
for cat, avg in zip(categories, averages):
    print(f"  {cat:<23} {avg:>15.4f}")


Category                   Avg Similarity
------------------------------------------
  Pos ↔ Pos                        0.9697
  Neg ↔ Neg                        0.9707
  Pos ↔ Neg                        0.9697
